In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add src to path
sys.path.append(str(Path('.').parent))
from src.dendritic_modeling.networks.layers.dendritic_conv2d import DendriteConv2d

# Set up visualization
plt.style.use('default')
torch.manual_seed(42)


ModuleNotFoundError: No module named 'src'

In [ ]:
# Create a test image with vertical and horizontal edges
image_size = 32
test_image = np.zeros((image_size, image_size))

# Add vertical edge in the middle
test_image[:, 14:18] = 1.0

# Add horizontal edge
test_image[8:12, :] = 0.5

# Add a diagonal feature
for i in range(20, 30):
    if i < image_size and i-10 < image_size:
        test_image[i, i-10] = 0.75

# Convert to tensor with batch and channel dimensions
# Shape: [batch=1, channels=1, height=32, width=32]
input_tensor = torch.tensor(test_image, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# Visualize the input
plt.figure(figsize=(6, 6))
plt.imshow(test_image, cmap='gray')
plt.colorbar(label='Pixel Value')
plt.title('Input Image')
plt.xlabel('Width')
plt.ylabel('Height')
plt.show()

print(f"Input tensor shape: {input_tensor.shape}")
print(f"Value range: [{input_tensor.min():.2f}, {input_tensor.max():.2f}]")


In [ ]:
# Create a DendriteConv2d layer
layer = DendriteConv2d(
    in_channels=1,      # Grayscale input
    out_channels=3,     # 3 different feature detectors
    kernel_size=3,      # 3x3 kernels
    stride=1,
    padding=1,          # Same padding to preserve size
    epsilon=1e-4
)

# Manually set the weights to create specific detectors
with torch.no_grad():
    # Filter 1: Vertical edge detector
    vertical_edge = torch.tensor([
        [1, 0, -1],
        [1, 0, -1],
        [1, 0, -1]
    ], dtype=torch.float32)
    
    # Filter 2: Horizontal edge detector
    horizontal_edge = torch.tensor([
        [1,  1,  1],
        [0,  0,  0],
        [-1, -1, -1]
    ], dtype=torch.float32)
    
    # Filter 3: Diagonal edge detector
    diagonal_edge = torch.tensor([
        [0, -1, -1],
        [1,  0, -1],
        [1,  1,  0]
    ], dtype=torch.float32)
    
    # Set excitatory weights (positive parts of filters)
    for i, kernel in enumerate([vertical_edge, horizontal_edge, diagonal_edge]):
        excitatory = torch.clamp(kernel, min=0)
        # Convert to log space for softplus parameterization
        layer.weight_E_raw.data[i, 0] = torch.log(excitatory + 1e-6)
    
    # Set inhibitory weights (negative parts of filters + baseline inhibition)
    for i, kernel in enumerate([vertical_edge, horizontal_edge, diagonal_edge]):
        inhibitory = torch.clamp(-kernel, min=0) + 0.1  # Add baseline inhibition
        layer.weight_I_raw.data[i, 0] = torch.log(inhibitory + 1e-6)

# Visualize the E and I kernels
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
filter_names = ['Vertical', 'Horizontal', 'Diagonal']

for i in range(3):
    # Excitatory weights
    e_weights = layer.weight_E[i, 0].detach().numpy()
    im1 = axes[0, i].imshow(e_weights, cmap='Reds', vmin=0)
    axes[0, i].set_title(f'{filter_names[i]} - E')
    axes[0, i].axis('off')
    plt.colorbar(im1, ax=axes[0, i], fraction=0.046)
    
    # Inhibitory weights
    i_weights = layer.weight_I[i, 0].detach().numpy()
    im2 = axes[1, i].imshow(i_weights, cmap='Blues', vmin=0)
    axes[1, i].set_title(f'{filter_names[i]} - I')
    axes[1, i].axis('off')
    plt.colorbar(im2, ax=axes[1, i], fraction=0.046)

plt.suptitle('Excitatory (E) and Inhibitory (I) Kernels', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Step 3a: Excitatory Convolution
# E = F.conv2d(input, weight_E, bias_E, ...)

with torch.no_grad():
    # Get the actual excitatory weights (after softplus transformation)
    weight_E = layer.weight_E
    
    # Perform excitatory convolution
    E = F.conv2d(
        input_tensor,
        weight_E,
        layer.bias_E,
        stride=layer.stride,
        padding=layer.padding
    )

print(f"Excitatory response shape: {E.shape}")
print(f"Excitatory response range: [{E.min():.3f}, {E.max():.3f}]")

# Visualize excitatory responses
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i in range(3):
    im = axes[i].imshow(E[0, i].numpy(), cmap='Reds')
    axes[i].set_title(f'{filter_names[i]} - E Response')
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046)
plt.suptitle('Step 3a: Excitatory Convolution Responses', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Step 3b: Inhibitory Convolution
# I = F.conv2d(input, weight_I, bias_I, ...)

with torch.no_grad():
    # Get the actual inhibitory weights (after softplus transformation)
    weight_I = layer.weight_I
    
    # Perform inhibitory convolution
    I = F.conv2d(
        input_tensor,
        weight_I,
        layer.bias_I,
        stride=layer.stride,
        padding=layer.padding
    )

print(f"Inhibitory response shape: {I.shape}")
print(f"Inhibitory response range: [{I.min():.3f}, {I.max():.3f}]")

# Visualize inhibitory responses
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i in range(3):
    im = axes[i].imshow(I[0, i].numpy(), cmap='Blues')
    axes[i].set_title(f'{filter_names[i]} - I Response')
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046)
plt.suptitle('Step 3b: Inhibitory Convolution Responses', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Step 3c: Divisive Normalization
# v_out = E / (E + I + epsilon)

with torch.no_grad():
    epsilon = layer.epsilon
    v_out = E / (E + I + epsilon)

print(f"Output shape: {v_out.shape}")
print(f"Output range: [{v_out.min():.3f}, {v_out.max():.3f}]")

# Visualize final output
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i in range(3):
    im = axes[i].imshow(v_out[0, i].numpy(), cmap='viridis')
    axes[i].set_title(f'{filter_names[i]} - Final Output')
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046)
plt.suptitle('Step 3c: Final Output after Divisive Normalization', fontsize=14)
plt.tight_layout()
plt.show()

# Compare with direct layer output
with torch.no_grad():
    layer_output = layer(input_tensor)
    
# Verify they match
print(f"\nVerification: Manual computation matches layer output: {torch.allclose(v_out, layer_output)}")


In [ ]:
# Analyze the effect of divisive normalization at specific locations
# Let's look at the vertical edge detector (filter 0)

filter_idx = 0  # Vertical edge detector

# Extract responses for this filter
E_response = E[0, filter_idx].numpy()
I_response = I[0, filter_idx].numpy()
final_output = v_out[0, filter_idx].numpy()

# Find interesting locations
# 1. Strong edge location (where vertical edge is present)
edge_y, edge_x = 16, 16  # Middle of vertical edge

# 2. Background location
bg_y, bg_x = 5, 5

# 3. Different feature location (horizontal edge)
other_y, other_x = 10, 10

locations = [
    (edge_y, edge_x, "Strong Edge"),
    (bg_y, bg_x, "Background"),
    (other_y, other_x, "Different Feature")
]

# Create detailed analysis figure
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

# Row 1: Show locations on images
axes[0, 0].imshow(test_image, cmap='gray')
axes[0, 0].set_title('Input Image')

axes[0, 1].imshow(E_response, cmap='Reds')
axes[0, 1].set_title('E Response')

axes[0, 2].imshow(I_response, cmap='Blues')
axes[0, 2].set_title('I Response')

axes[0, 3].imshow(final_output, cmap='viridis')
axes[0, 3].set_title('Final Output')

# Mark the locations
for i in range(4):
    for y, x, label in locations:
        color = ['red', 'blue', 'green'][locations.index((y, x, label))]
        axes[0, i].plot(x, y, 'o', color=color, markersize=10, 
                       markeredgecolor='white', markeredgewidth=2)

# Row 2 & 3: Bar plots for each location
for row, (y, x, label) in enumerate(locations):
    e_val = E_response[y, x]
    i_val = I_response[y, x]
    out_val = final_output[y, x]
    
    # Calculate what the output would be without normalization
    linear_output = e_val - i_val
    
    # Bar plot
    ax = axes[row//2 + 1, (row%2)*2]
    bars = ax.bar(['E', 'I', 'E+I', 'Output'], 
                   [e_val, i_val, e_val + i_val, out_val],
                   color=['red', 'blue', 'purple', 'green'])
    ax.set_title(f'{label} (y={y}, x={x})')
    ax.set_ylabel('Response Value')
    
    # Show the normalization equation
    ax_text = axes[row//2 + 1, (row%2)*2 + 1]
    ax_text.axis('off')
    equation_text = f"""Location: {label}
    
E = {e_val:.3f}
I = {i_val:.3f}
ε = {epsilon:.4f}

Output = E / (E + I + ε)
       = {e_val:.3f} / ({e_val:.3f} + {i_val:.3f} + {epsilon:.4f})
       = {e_val:.3f} / {e_val + i_val + epsilon:.3f}
       = {out_val:.3f}

Without normalization (E - I):
       = {e_val:.3f} - {i_val:.3f} = {linear_output:.3f}

Compression factor: {abs(linear_output / (out_val + 1e-8)):.2f}x"""
    
    ax_text.text(0.05, 0.95, equation_text, transform=ax_text.transAxes,
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Divisive Normalization Analysis at Different Locations', fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Create a more complex test image with natural-like features
complex_image = np.zeros((64, 64))

# Add multiple features
# Vertical stripes
for i in range(0, 64, 8):
    complex_image[:, i:i+3] = np.random.uniform(0.5, 1.0)

# Horizontal stripes
for i in range(0, 64, 12):
    complex_image[i:i+4, :] *= 0.7

# Add some gaussian blobs
from scipy.ndimage import gaussian_filter
noise = np.random.randn(64, 64) * 0.3
complex_image += gaussian_filter(noise, sigma=3)

# Normalize to [0, 1]
complex_image = (complex_image - complex_image.min()) / (complex_image.max() - complex_image.min())

# Convert to tensor
complex_tensor = torch.tensor(complex_image, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# Process through the layer
with torch.no_grad():
    # Get individual components
    E_complex = F.conv2d(complex_tensor, layer.weight_E, layer.bias_E, 
                        stride=layer.stride, padding=layer.padding)
    I_complex = F.conv2d(complex_tensor, layer.weight_I, layer.bias_I,
                        stride=layer.stride, padding=layer.padding)
    output_complex = layer(complex_tensor)

# Visualize the processing pipeline
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

# Row 1: Input
axes[0, 0].imshow(complex_image, cmap='gray')
axes[0, 0].set_title('Input Image', fontsize=12)
axes[0, 0].axis('off')

for i in range(1, 4):
    axes[0, i].axis('off')

# Rows 2-4: Each filter's response
for filter_idx in range(3):
    row = filter_idx + 1
    
    # E response
    e_resp = E_complex[0, filter_idx].numpy()
    im1 = axes[row, 0].imshow(e_resp, cmap='Reds')
    axes[row, 0].set_title(f'{filter_names[filter_idx]} - E', fontsize=12)
    axes[row, 0].axis('off')
    
    # I response
    i_resp = I_complex[0, filter_idx].numpy()
    im2 = axes[row, 1].imshow(i_resp, cmap='Blues')
    axes[row, 1].set_title(f'{filter_names[filter_idx]} - I', fontsize=12)
    axes[row, 1].axis('off')
    
    # E+I (denominator)
    denominator = e_resp + i_resp + layer.epsilon
    im3 = axes[row, 2].imshow(denominator, cmap='Purples')
    axes[row, 2].set_title(f'{filter_names[filter_idx]} - E+I+ε', fontsize=12)
    axes[row, 2].axis('off')
    
    # Final output
    final = output_complex[0, filter_idx].numpy()
    im4 = axes[row, 3].imshow(final, cmap='viridis')
    axes[row, 3].set_title(f'{filter_names[filter_idx]} - Output', fontsize=12)
    axes[row, 3].axis('off')

plt.suptitle('DendriteConv2d Processing Pipeline on Complex Image', fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Property 1: Output is always bounded between 0 and 1
print("Property 1: Bounded Output")
print(f"Output range: [{output_complex.min():.3f}, {output_complex.max():.3f}]")
print("This is guaranteed by the divisive normalization formula: E/(E+I+ε) ∈ [0,1]\n")

# Property 2: Contrast gain control
print("Property 2: Contrast Gain Control")
# Test with different contrast levels
contrasts = [0.25, 0.5, 1.0, 2.0, 4.0]
responses = []

with torch.no_grad():
    for contrast in contrasts:
        scaled_input = complex_tensor * contrast
        output = layer(scaled_input)
        # Measure average response magnitude
        avg_response = output.abs().mean().item()
        responses.append(avg_response)

# Plot contrast response
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Dendritic response
ax1.plot(contrasts, responses, 'o-', linewidth=2, markersize=8, label='DendriteConv2d')
ax1.set_xlabel('Input Contrast', fontsize=12)
ax1.set_ylabel('Average Response', fontsize=12)
ax1.set_title('Contrast Response Function', fontsize=14)
ax1.grid(True, alpha=0.3)

# Compare with linear response
linear_responses = [r * c for r, c in zip([responses[2]] * len(contrasts), 
                                          [c/contrasts[2] for c in contrasts])]
ax1.plot(contrasts, linear_responses, '--', linewidth=2, label='Linear (no normalization)')
ax1.legend()

# Show saturation effect
ax2.plot(contrasts, [r/c for r, c in zip(responses, contrasts)], 'o-', linewidth=2, markersize=8)
ax2.set_xlabel('Input Contrast', fontsize=12)
ax2.set_ylabel('Response / Contrast (Gain)', fontsize=12)
ax2.set_title('Automatic Gain Control', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nThe divisive normalization provides automatic gain control:")
print("- At low contrasts, the gain is high (amplification)")
print("- At high contrasts, the gain decreases (compression)")
print("- This helps maintain sensitivity across different input ranges\n")


In [ ]:
# Create a summary visualization
fig = plt.figure(figsize=(16, 10))

# Create grid for subplots
gs = fig.add_gridspec(3, 4, width_ratios=[1, 1, 1, 1.5], height_ratios=[1, 1, 1])

# Original image
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(complex_image, cmap='gray')
ax1.set_title('1. Input Image', fontsize=14, fontweight='bold')
ax1.axis('off')

# Show one filter's processing
filter_idx = 0  # Vertical edge detector

# E weights
ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(layer.weight_E[filter_idx, 0].detach().numpy(), cmap='Reds')
ax2.set_title('2. E Kernel', fontsize=14, fontweight='bold')
ax2.axis('off')

# I weights
ax3 = fig.add_subplot(gs[0, 2])
ax3.imshow(layer.weight_I[filter_idx, 0].detach().numpy(), cmap='Blues')
ax3.set_title('3. I Kernel', fontsize=14, fontweight='bold')
ax3.axis('off')

# E response
ax4 = fig.add_subplot(gs[1, 0])
ax4.imshow(E_complex[0, filter_idx].numpy(), cmap='Reds')
ax4.set_title('4. E Response', fontsize=14, fontweight='bold')
ax4.axis('off')

# I response
ax5 = fig.add_subplot(gs[1, 1])
ax5.imshow(I_complex[0, filter_idx].numpy(), cmap='Blues')
ax5.set_title('5. I Response', fontsize=14, fontweight='bold')
ax5.axis('off')

# Final output
ax6 = fig.add_subplot(gs[1, 2])
ax6.imshow(output_complex[0, filter_idx].numpy(), cmap='viridis')
ax6.set_title('6. Final Output', fontsize=14, fontweight='bold')
ax6.axis('off')

# Text explanation
ax_text = fig.add_subplot(gs[:, 3])
ax_text.axis('off')

explanation = """DendriteConv2d Processing Steps:

1. Input Image
   • Shape: [batch, channels, height, width]
   • Values: Any real numbers

2. Excitatory Convolution
   • E = F.conv2d(input, weight_E, bias_E)
   • weight_E = F.softplus(weight_E_raw)
   • Captures positive features

3. Inhibitory Convolution
   • I = F.conv2d(input, weight_I, bias_I)
   • weight_I = F.softplus(weight_I_raw)
   • Captures negative features + baseline

4. Divisive Normalization
   • v_out = E / (E + I + ε)
   • ε prevents division by zero
   • Output always in [0, 1]

Key Properties:
✓ Bounded output: [0, 1]
✓ Contrast gain control
✓ Biologically inspired
✓ Robust to input variations
✓ Interpretable E/I balance

Biological Analogy:
• E: Excitatory synapses
• I: Inhibitory synapses
• Divisive normalization: 
  Shunting inhibition in dendrites"""

ax_text.text(0.05, 0.95, explanation, transform=ax_text.transAxes,
            fontsize=12, verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

# Add flow arrows
arrow_props = dict(arrowstyle='->', lw=3, color='black')
# Input to E/I kernels
ax1.annotate('', xy=(0.15, 0.5), xytext=(-0.15, 0.5),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=arrow_props, annotation_clip=False)

plt.suptitle('DendriteConv2d: Complete Processing Pipeline', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()
